In [79]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql.functions import col

In [131]:
spark.stop()

In [132]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

### Question 1

In [81]:
spark

In [59]:
spark.version

'3.3.2'

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-25 03:59:10--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.239.192.40, 54.239.192.95, 54.239.192.206, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.239.192.40|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: 'yellow_tripdata_2024-10.parquet.2'

     0K .......... .......... .......... .......... ..........  0% 2.70M 23s
    50K .......... .......... .......... .......... ..........  0% 81.0M 12s
   100K .......... .......... .......... .......... ..........  0%  103M 8s
   150K .......... .......... .......... .......... ..........  0% 55.0M 6s
   200K .......... .......... .......... .......... ..........  0% 10.1M 6s
   250K .......... .......... .......... .......... ..........  0% 11.0M 6s
   300K .......... .......... .......... .......... .......... 

In [5]:
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-08 23:35:34--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.187, 18.155.128.222, 18.155.128.6, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.187|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  89.1MB/s    in 0.7s    

2025-03-08 23:35:35 (89.1 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [4]:
!ls -lh yellow_tripdata_2024-10.parquet

-rw-r--r-- 1 Abderrahmen Mansour 197609 62M Dec 18 22:21 yellow_tripdata_2024-10.parquet


In [82]:
df = spark.read.parquet("yellow_tripdata_2024-10.parquet")

In [6]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [11]:
# df = df.repartition(4)
# df.write.parquet("yellow_2024_10_partitioned")

In [7]:
df = df.repartition(4)
df.write.mode("overwrite").parquet("yellow_2024_10_partitioned")

### Question 2

In [10]:
!ls -lh yellow_2024_10_partitioned/

total 97M
-rw-r--r-- 1 Abderrahmen Mansour 197609   0 Mar 25 04:34 _SUCCESS
-rw-r--r-- 1 Abderrahmen Mansour 197609 25M Mar 25 04:34 part-00000-41b26c2a-b142-46c7-ad44-6ce0cada30b0-c000.snappy.parquet
-rw-r--r-- 1 Abderrahmen Mansour 197609 25M Mar 25 04:34 part-00001-41b26c2a-b142-46c7-ad44-6ce0cada30b0-c000.snappy.parquet
-rw-r--r-- 1 Abderrahmen Mansour 197609 25M Mar 25 04:34 part-00002-41b26c2a-b142-46c7-ad44-6ce0cada30b0-c000.snappy.parquet
-rw-r--r-- 1 Abderrahmen Mansour 197609 25M Mar 25 04:34 part-00003-41b26c2a-b142-46c7-ad44-6ce0cada30b0-c000.snappy.parquet


### Question 3

In [13]:
from pyspark.sql.functions import to_date
from pyspark.sql import functions as F

In [11]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-10-07 18:40:43|  2024-10-07 20:10:56|              1|         14.8|        99|                 N|         127|         225|           1|       47.5|  0.0|    0.5|       0.

In [16]:
df.withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)).select('pickup_date').show()

+-----------+
|pickup_date|
+-----------+
| 2024-10-02|
| 2024-10-10|
| 2024-10-03|
| 2024-10-09|
| 2024-10-09|
| 2024-10-08|
| 2024-10-02|
| 2024-10-02|
| 2024-10-02|
| 2024-10-02|
| 2024-10-05|
| 2024-10-02|
| 2024-10-04|
| 2024-10-08|
| 2024-10-06|
| 2024-10-10|
| 2024-10-02|
| 2024-10-03|
| 2024-10-02|
| 2024-10-09|
+-----------+
only showing top 20 rows



In [63]:

df_filtered = df.filter(to_date(df.tpep_pickup_datetime) == "2024-10-15")

In [20]:
df_filtered.select('tpep_pickup_datetime').show()

+--------------------+
|tpep_pickup_datetime|
+--------------------+
| 2024-10-15 12:40:01|
| 2024-10-15 15:20:29|
| 2024-10-15 17:56:35|
| 2024-10-15 17:56:35|
| 2024-10-15 17:37:15|
| 2024-10-15 20:22:03|
| 2024-10-15 13:51:03|
| 2024-10-15 15:39:41|
| 2024-10-15 00:38:51|
| 2024-10-15 18:32:10|
| 2024-10-15 22:27:05|
| 2024-10-15 21:42:43|
| 2024-10-15 15:50:49|
| 2024-10-15 22:07:27|
| 2024-10-15 20:55:48|
| 2024-10-15 12:09:26|
| 2024-10-15 09:45:19|
| 2024-10-15 01:29:37|
| 2024-10-15 16:30:46|
| 2024-10-15 01:23:09|
+--------------------+
only showing top 20 rows



In [21]:
df_filtered.count()

125567

### Question 4

In [ ]:
df = df.withColumn("trip_duration_hours", \
                   (col("tpep_dropoff_datetime") - col("tpep_pickup_datetime")) / 3600)
longest_trip = df.selectExpr("MAX(trip_duration_hours) AS max_duration").collect()[0]["max_duration"]

In [23]:
longest_trip

datetime.timedelta(seconds=162, microseconds=617778)

### Question 5:

In [40]:
# http://localhost:4040/

### Question 6

In [25]:
! wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-25 04:54:14--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.239.192.206, 54.239.192.100, 54.239.192.95, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.239.192.206|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: 'taxi_zone_lookup.csv.1'

     0K .......... ..                                         100% 81.2M=0s

2025-03-25 04:54:15 (81.2 MB/s) - 'taxi_zone_lookup.csv.1' saved [12331/12331]



In [65]:
!ls -lh taxi_zone_lookup.csv

-rw-r--r-- 1 Abderrahmen Mansour 197609 13K Feb 22  2024 taxi_zone_lookup.csv


In [83]:
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True)

In [67]:
df_zones.printSchema()

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [68]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [69]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 02:30:44|  2024-10-01 02:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [84]:
df.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [135]:
import pyspark
print(pyspark.__version__)  # Should match your Spark version (3.3.2)

import py4j
print(py4j.__version__)  # Should be compatible with Spark


3.5.5
0.10.9.7


In [136]:
! pip install --upgrade py4j

  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyspark 3.5.5 requires py4j==0.10.9.7, but you have py4j 0.10.9.9 which is incompatible.


In [ ]:
! pip uninstall pyspark

In [ ]:
! pip install pyspark

^C
  Using cached py4j-0.10.9.7-py2.py3-none-any.whl.metadata (1.5 kB)
Using cached py4j-0.10.9.7-py2.py3-none-any.whl (200 kB)
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9


In [ ]:
print(spark._jvm)  # Should not be None

In [ ]:
print(spark._jsparkSession)  # Should return a Java object

In [ ]:
pyspark

In [ ]:
spark.sql("SELECT 1").show()

In [34]:
result = spark.sql("""
    SELECT .VendorID , t.PULocationID , z.LocationID , z.Zone
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    LIMIT 10
""")

result.show()

Py4JError: An error occurred while calling o25.sql. Trace:
py4j.Py4JException: Method sql([class java.lang.String, class [Ljava.lang.Object;]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:318)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:326)
	at py4j.Gateway.invoke(Gateway.java:274)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Thread.java:834)



In [33]:
result = spark.sql("""
    SELECT z.Zone, COUNT(t.PULocationID) AS pickup_count
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY pickup_count ASC
    LIMIT 1
""")

result.show()

Py4JError: An error occurred while calling o25.sql. Trace:
py4j.Py4JException: Method sql([class java.lang.String, class [Ljava.lang.Object;]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:318)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:326)
	at py4j.Gateway.invoke(Gateway.java:274)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Thread.java:834)

